# BERT and Job Categories - Exercise

**Business problem:** help a publisher choose a job category by suggesting one label or a shortlist of three.

We use the real job ads behind page 37 of the Week 08 NLP lecture. A model supports a review decision; it does not make the business decision for us.

## Learning objectives

- Compare a text representation with frequency and TF-IDF baselines.
- Keep fitting, validation selection and final evaluation separate.
- Interpret errors and limits before proposing a business use.

## Dataset and practical scope

The supplied `nlp_real_job_categories.csv.gz` contains **5,011 real, cleaned, redacted job descriptions across 143 categories**, not synthetic examples. It is the exact classification cohort behind page 37 of `Slides_MSC_Data_Science_Week_08_EN.pdf`.

- **Train:** 3,328 ads from July and August 2026; 139 categories.
- **Validation:** 918 ads from 1-14 September; 118 categories.
- **Test:** 765 ads from 15 September onward; 114 categories.

The original 30,000-ad sample was filtered to one `JOBS:` category per ad; conflicting labels and later exact duplicate descriptions were removed. Cleaning and the chronological split are already supplied, so we focus on modelling rather than repeat data preparation. IDs and dates are not model features.

Some later categories are absent from training: **three validation ads and two test ads** have unseen labels. Keep them in evaluation; a classifier cannot suggest a category it never learned. Exact duplicate removal does not remove every near-duplicate or template. This is a previously inspected teaching cohort, not a pristine research test.

The instructor confirmed permission to include this classification subset for the repository's audience. Redaction is not a guarantee of anonymity; do not assume unrestricted redistribution rights. See [data notes](../../../00_Data/nlp_data_readme.md). Run the notebook from top to bottom; full feature extraction is practical work, not an extra lecture segment.

Complete the two marked coding gaps before running their cells.

## Libraries and settings

Use the prepared Python 3.11 Codespace kernel. No GPU or API key is needed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 7
np.random.seed(SEED)
DATA_ROOT = Path('../../../00_Data')

## 1. Minimal data inspection

Pandas reads the compressed CSV directly. Preserve the supplied chronological partitions. We inspect counts and date ranges rather than print full descriptions.

In [ ]:
jobs = pd.read_csv(DATA_ROOT / 'nlp_real_job_categories.csv.gz',
                   parse_dates=['creation_date'])
required = {'teaching_id', 'creation_date', 'text', 'category', 'split'}
if not required.issubset(jobs.columns) or jobs[list(required)].isna().any().any():
    raise ValueError('Missing columns or values in the prepared real job ads.')
if not jobs['split'].isin(['train', 'validation', 'test']).all():
    raise ValueError('Unexpected partition name.')
if not jobs['teaching_id'].is_unique or jobs['text'].str.casefold().duplicated().any():
    raise ValueError('Repeated IDs or cleaned descriptions in the prepared data.')
parts = {name: jobs[jobs['split'] == name].copy()
         for name in ['train', 'validation', 'test']}
if any(frame.empty for frame in parts.values()):
    raise ValueError('Every partition needs ads.')
if not (parts['train']['creation_date'].max() < parts['validation']['creation_date'].min()
        <= parts['validation']['creation_date'].max() < parts['test']['creation_date'].min()):
    raise ValueError('The chronological partitions overlap or are out of order.')
training_categories = set(parts['train']['category'])
display(pd.DataFrame([
    {'split': name, 'ads': len(frame), 'categories': frame['category'].nunique(),
     'first date': frame['creation_date'].min().date(),
     'last date': frame['creation_date'].max().date(),
     'ads with unseen category': (~frame['category'].isin(training_categories)).sum()}
    for name, frame in parts.items()
]))
display(parts['train']['category'].value_counts().head(10).rename('training ads').to_frame())

## 2. Business and statistical baseline

A frequency baseline ignores text and suggests common training categories. TF-IDF plus logistic regression is the lexical baseline. Its vocabulary and IDF are fitted on training ads only, after BERT tokenization below gives both learned classifiers the same retained input.

The supplied evaluation helper reports accuracy, macro-F1 and top-3 hit rate. Macro-F1 averages the categories actually present in the evaluated split; predicted-only labels do not add categories to that average. If a TF-IDF row has no known terms, request manual review and count it as a miss, rather than silently use an intercept-only suggestion.

In [ ]:
import re
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

y_train = parts['train']['category'].to_numpy()
y_validation = parts['validation']['category'].to_numpy()
y_test = parts['test']['category'].to_numpy()
prior = DummyClassifier(strategy='prior', random_state=SEED).fit(
    np.ones((len(y_train), 1)), y_train)

TOKEN_PATTERN = re.compile(r"[^\W\d_]+(?:['\u2019-][^\W\d_]+)*", re.UNICODE)

def tokenize(text):
    return [word for word in TOKEN_PATTERN.findall(text) if len(word) > 1]

tfidf = TfidfVectorizer(
    tokenizer=tokenize, token_pattern=None, min_df=3, max_features=20_000,
    sublinear_tf=True, dtype=np.float32)

def evaluate_suggestions(labels, probabilities, classes, covered):
    if probabilities.shape != (len(labels), len(classes)) or covered.shape != (len(labels),):
        raise ValueError('Predictions, labels and coverage have incompatible shapes.')
    if not np.isfinite(probabilities).all() or not np.allclose(probabilities.sum(axis=1), 1):
        raise ValueError('Expected finite classifier probabilities summing to one.')
    ranking = np.argsort(-probabilities, axis=1, kind='stable')
    predicted = np.where(covered, classes[ranking[:, 0]], '__REVIEW__')
    top3 = classes[ranking[:, :3]]
    hits = (top3 == labels[:, None]).any(axis=1) & covered
    metrics = {
        'accuracy': accuracy_score(labels, predicted),
        'macro-F1': f1_score(labels, predicted, labels=np.unique(labels),
                             average='macro', zero_division=0),
        'top-3 hit rate': hits.mean(),
        'top-3 hits': int(hits.sum()),
        'review only': int((~covered).sum()),
    }
    return metrics, predicted, top3

## 3. Reuse a pretrained encoder

First use downloads public BERT files (about 700 MB). Job text is processed locally, not sent to a prediction API. BERT's parameters stay fixed: only the later category classifier learns from our labels. Download the weights before class.

We use the same pinned multilingual cased BERT checkpoint as the lecture. Each ad is limited to **126 content pieces plus two BERT markers**. TF-IDF receives a decoded version of those same pieces, not the full description. This keeps the retained input comparable, although decoding changes formatting. The next cell reports how many descriptions were truncated.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

torch.manual_seed(SEED)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
MODEL = 'google-bert/bert-base-multilingual-cased'
REVISION = '3f076fdb1ab68d5b2880cb87a0886f315b8146f8'
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
encoder = AutoModelForMaskedLM.from_pretrained(
    MODEL, revision=REVISION, use_safetensors=True).eval()
encoder.requires_grad_(False)
if tokenizer.do_lower_case:
    raise ValueError('This example requires the cased multilingual checkpoint.')
print('Frozen encoder; pooled values per document:', encoder.config.hidden_size)

## 4. Pool contextual token features

BERT gives each token (a word or word piece) a vector of **768 numbers**. Each vector depends on the surrounding text. But our category classifier needs **one fixed-length vector per ad**, even when ads have different lengths.

**Pooling means combining these token vectors into one ad vector.** Here, we simply average them, one coordinate at a time. For example, 80 content tokens give 80 vectors; averaging them gives one vector of 768 numbers, not a vector of 80 numbers.

A tiny made-up example with only three numbers per token:

| Token | Vector |
|---|---|
| software | [1, 2, 0] |
| engineer | [3, 0, 2] |
| wanted | [2, 1, 1] |
| **Average for the ad** | **[2, 1, 1]** |

**Coding task:** calculate this average in the `pooled` line below:
- `hidden` contains BERT's token vectors.
- `content` marks real text with 1 and padding or special markers with 0. Multiplying by it leaves out those extra positions.
- Sum over token positions (`dim=1`), then divide by `content.sum(dim=1)`, the number of content tokens.

`job ad -> BERT token vectors -> average -> 768-number ad vector -> classifier -> category suggestions`

**This step does not train BERT or predict a category.** It prepares the input for the classifier we train next. BERT's numbers stay fixed, including when we process validation and test ads; TF-IDF and scaling still learn only from training data. The code prints progress as it processes each partition.

**Limitations:** averaging loses token-level detail. We also retain only the first **126 content pieces** of each ad, so the vector may miss requirements near the end; it does not necessarily summarize the whole description.

In [ ]:
def encode_ads(texts):
    rows, retained = [], []
    truncated = 0
    for start in range(0, len(texts), 8):
        examples = list(texts[start:start + 8])
        truncated += sum(len(tokenizer.tokenize(text, verbose=False)) > 126
                         for text in examples)
        batch = tokenizer(examples, padding=True, truncation=True, max_length=128,
                          return_tensors='pt', return_special_tokens_mask=True)
        special = batch.pop('special_tokens_mask')
        content = (batch['attention_mask'] * (1 - special)).unsqueeze(-1)
        if torch.any(content.sum(dim=1) == 0):
            raise ValueError('An ad has no content tokens after tokenization.')
        with torch.inference_mode():
            hidden = encoder.base_model(**batch).last_hidden_state
            pooled = None  # TODO: masked mean over token positions
        rows.append(pooled.cpu().numpy())
        retained.extend(tokenizer.batch_decode(batch['input_ids'], skip_special_tokens=True))
        if start % 160 == 0:
            print(f'  Encoded {min(start + 8, len(texts)):,}/{len(texts):,} ads', flush=True)
    return np.vstack(rows), retained, truncated

vectors, retained, truncation_rows = {}, {}, []
for name, frame in parts.items():
    print(f'Encoding {name}: {len(frame):,} ads', flush=True)
    vectors[name], retained[name], truncated = encode_ads(frame['text'].tolist())
    truncation_rows.append({'split': name, 'ads': len(frame), 'truncated ads': truncated})
lexical = {'train': tfidf.fit_transform(retained['train'])}
lexical.update({name: tfidf.transform(retained[name]) for name in ['validation', 'test']})
coverage = {
    'TF-IDF': {name: matrix.getnnz(axis=1) > 0 for name, matrix in lexical.items()},
    'Frozen mBERT': {name: np.ones(len(frame), dtype=bool) for name, frame in parts.items()},
}
display(pd.DataFrame(truncation_rows))
print('BERT feature shapes:', {name: matrix.shape for name, matrix in vectors.items()})

### Interpret the result

Does frozen BERT produce identical outputs for different texts?

**Your interpretation:** Replace this sentence with your answer.

## 5. Train and select the category classifier

**Coding task:** create `LogisticRegression(C=regularization, class_weight='balanced', max_iter=2000, random_state=SEED)`.

The loop compares C=0.3 and C=1 using validation macro-F1; choose the smaller C on a tie. Class balancing uses training-label frequencies to give rare categories more weight. Scaling is fitted on training BERT vectors only. These choices match the lecture recipe; neither classifier is refitted on validation data.

In [ ]:
representations = {'TF-IDF': lexical, 'Frozen mBERT': vectors}
chosen = {}
validation_rows = []
for name, matrices in representations.items():
    candidates = []
    for regularization in [0.3, 1.0]:
        classifier = None  # TODO: balanced logistic regression with the current C
        model = (make_pipeline(StandardScaler(), classifier)
                 if name == 'Frozen mBERT' else classifier)
        model.fit(matrices['train'], y_train)
        result, _, _ = evaluate_suggestions(
            y_validation, model.predict_proba(matrices['validation']),
            model.classes_, coverage[name]['validation'])
        candidates.append((result['macro-F1'], regularization, model))
        validation_rows.append({'model': name, 'C': regularization,
                                'validation macro-F1': result['macro-F1']})
    candidates.sort(key=lambda item: (-item[0], item[1]))
    chosen[name] = candidates[0][2]
display(pd.DataFrame(validation_rows))

## 6. Evaluate fixed choices

Do not change settings after seeing test results.

- **Accuracy:** was the first suggested category correct?
- **Macro-F1:** balance precision and recall per category, then give each category present in the test set equal weight.
- **Top-3 hit rate:** was the one true category among three suggestions? This is not precision@3.

Every test ad counts, including unseen labels and any review-only cases. Stable ordering breaks equal-probability ties by the classifier's class order, matching the lecture.

In [ ]:
test_predictions, test_shortlists = {}, {}
results = []
for name, model in {'Frequency': prior, **chosen}.items():
    matrix = (np.ones((len(y_test), 1)) if name == 'Frequency'
              else representations[name]['test'])
    covered = (np.ones(len(y_test), dtype=bool) if name == 'Frequency'
               else coverage[name]['test'])
    result, prediction, top3 = evaluate_suggestions(
        y_test, model.predict_proba(matrix), model.classes_, covered)
    results.append({'model': name, **result})
    test_predictions[name], test_shortlists[name] = prediction, top3
results = pd.DataFrame(results).set_index('model')
display(results.round(3))
print(f'All {len(y_test)} test ads count, including '
      f'{int((~parts["test"]["category"].isin(training_categories)).sum())} '
      'ads whose category was absent from training.')

### Interpret the result

Which recipe would you investigate further? What prevents a deployment claim?

**Your interpretation:** Cite your measured results, one limitation and one business outcome to evaluate.

## Interpretation: inspect individual decisions

With more than a hundred categories, a full confusion matrix is hard to read. Compare first-choice accuracy with shortlist hit rate, then inspect a few BERT errors by teaching ID and category. Use the corresponding local description if needed; avoid publishing additional text extracts. Category probabilities are not automatically calibrated.

In [ ]:
results[['accuracy', 'top-3 hit rate']].plot.bar(
    ylim=(0, 1), rot=0, figsize=(7, 4), ylabel='Share of test ads',
    title='One category versus a three-category shortlist')
plt.tight_layout()
plt.show()
prediction = test_predictions['Frozen mBERT']
decisions = parts['test'][['teaching_id', 'category']].copy()
decisions['predicted'] = prediction
decisions['top-3 suggestions'] = [' | '.join(row) for row in test_shortlists['Frozen mBERT']]
decisions['correct first suggestion'] = prediction == y_test
display(decisions.loc[~decisions['correct first suggestion']].head(6))
print('BERT first-suggestion errors:', int((prediction != y_test).sum()))

## Optional: a masked-token prediction

The output below is actual pretrained-model inference, not a training update or job-category prediction. The original text token, not the highest prediction, would be the MLM training target.

In [ ]:
masked = tokenizer('We are hiring a [MASK] engineer.', return_tensors='pt')
position = int((masked['input_ids'][0] == tokenizer.mask_token_id).nonzero()[0].item())
with torch.inference_mode():
    probabilities = encoder(**masked).logits[0, position].softmax(dim=-1)
top = probabilities.topk(5)
display(pd.DataFrame({'token': tokenizer.convert_ids_to_tokens(top.indices.tolist()),
                      'model probability': top.values.tolist()}))

## Summary and business implications

Select with validation evidence, evaluate fixed recipes, then connect errors to publisher review. Changing representations is not a controlled causal experiment.

## Limitations

This real-data teaching cohort has already informed lecture development; it is not an untouched research benchmark. The duplicate/conflict audit used the full period retrospectively; near-duplicates and templates remain. Categories are imbalanced, and two test ads have categories absent from training. In the lecture, 757 of 765 test descriptions were truncated.

Raw BERT mean pooling is not a retrieval-trained sentence embedding. Comparing externally pretrained BERT with train-only TF-IDF does not isolate a causal architecture effect. Software versions may change measured results. Test review usefulness, future periods, privacy and category-taxonomy changes before operational use.

## Further exercises

- Inspect one wrong first suggestion: does the shortlist still contain the correct category?
- Explain why the two unseen-category test ads must remain in the denominator.
- What evidence might truncation remove, and how would you test a longer input without choosing settings on the test set?
- Propose a future-period evaluation and a measure of publisher review effort.

## References

**Required reading:** BERT overview: https://huggingface.co/docs/transformers/model_doc/bert
**Instructor references:** Vaswani et al. (2017), *Attention Is All You Need*, https://arxiv.org/abs/1706.03762;
Devlin et al. (2019), *BERT*, https://aclanthology.org/N19-1423/;
Brown et al. (2020), *Language Models are Few-Shot Learners*, https://arxiv.org/abs/2005.14165.

**Dataset provenance and distribution:** [NLP data notes](../../../00_Data/nlp_data_readme.md). **Lecture reference:** `Slides_MSC_Data_Science_Week_08_EN.pdf`, page 37 (original real-data comparison).

### Jupyter notebook - footer info

Include your name, submission date and a short description of your changes when submitting.